# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.18.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 3.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.3 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.8 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-25 00:19:21.879127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779668362.082267      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779668362.142489      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779668362.623332      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779668362.623371      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779668362.623373      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [ ]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Qwen3-4B"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Qwen3-4B-GPTQ-4bit"
TOKENIZER_ID = "Qwen/Qwen3-4B"

MODEL_NAME           = "Qwen3-4B"
MODEL_NAME_IN_REPO   = "Qwen3-4B-GPTQ"
COMPRESSION_METHOD   = "GPTQ"
MODEL_TYPE           = "Quantization"
#SPARSITY = 70
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
"""local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)"""

'local_path = snapshot_download(\n    repo_id=MODEL_ID,\n    allow_patterns=f"{PATH}/*",\n    local_dir="/kaggle/working/model"\n)'

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [ ]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-25 00:19:49 [utils.py:233] non-default args: {'tokenizer': 'meta-llama/Llama-3.2-3B-Instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': 'EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-25 00:20:10 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 05-25 00:20:10 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 05-25 00:20:10 [model.py:1582] Using max model len 256
INFO 05-25 00:20:10 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 05-25 00:20:11 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-25 00:20:11 [vllm.py:754] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

WARNING 05-25 00:20:12 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(Worker pid=264) INFO 05-25 00:20:51 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=264) WARNING 05-25 00:20:52 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=265) WARNING 05-25 00:20:52 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=265) WARNING 05-25 00:20:52 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=264) WARNING 05-25 00:20:52 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=265) INFO 05-25 00:20:52 [parallel_state.py:1717] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP ra

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.08s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.08s/it]
(Worker_TP0 pid=264) 


(Worker_TP0 pid=264) INFO 05-25 00:21:06 [default_loader.py:384] Loading weights took 1.17 seconds
(Worker_TP0 pid=264) INFO 05-25 00:21:07 [gpu_model_runner.py:4566] Model loading took 1.08 GiB memory and 13.146898 seconds
(Worker_TP0 pid=264) INFO 05-25 00:21:20 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/2183e37a03/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=264) INFO 05-25 00:21:20 [backends.py:1048] Dynamo bytecode transform time: 12.36 s
(Worker_TP1 pid=265) INFO 05-25 00:21:27 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=264) INFO 05-25 00:21:27 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=264) INFO 05-25 00:21:33 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 11.96 s
(Worker_TP0 pid=264) INFO 05-25 00:21:35 [decorators.py:627] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/47f

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.37it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.37it/s]


(Worker_TP1 pid=265) INFO 05-25 00:22:06 [gpu_worker.py:617] CUDA graph pool memory: 1.0 GiB (actual), 1.09 GiB (estimated), difference: 0.09 GiB (9.0%).
(Worker_TP0 pid=264) INFO 05-25 00:22:06 [gpu_model_runner.py:5746] Graph capturing finished in 13 secs, took 1.00 GiB
(Worker_TP0 pid=264) INFO 05-25 00:22:06 [gpu_worker.py:617] CUDA graph pool memory: 1.0 GiB (actual), 1.09 GiB (estimated), difference: 0.09 GiB (9.0%).
(EngineCore pid=239) INFO 05-25 00:22:06 [core.py:281] init engine (profile, create kv cache, warmup model) took 59.51 seconds
(EngineCore pid=239) INFO 05-25 00:22:08 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 05-25 00:22:08 [llm.py:391] Supported tasks: ['generate']
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times      = []
latency_times   = []
decode_times    = []
inference_times = []  # prefill + decode

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    ttft      = metrics.first_token_ts - metrics.scheduled_ts
    latency   = metrics.last_token_ts  - metrics.queued_ts
    decode    = metrics.last_token_ts  - metrics.first_token_ts
    inference = ttft + decode                                   # prefill + decode

    ttft_times.append(ttft)
    latency_times.append(latency)
    decode_times.append(decode)
    inference_times.append(inference)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr, latency_arr, decode_arr, inference_arr = (
    np.array(ttft_times),
    np.array(latency_times),
    np.array(decode_times),
    np.array(inference_times),
)

ttft_mean,      ttft_std      = ttft_arr.mean(),      ttft_arr.std()
latency_mean,   latency_std   = latency_arr.mean(),   latency_arr.std()
inference_mean, inference_std = inference_arr.mean(), inference_arr.std()

# decode throughput: excludes first token, over decode-only window
decode_throughput_arr                           = (MAX_GENERATION_TOKENS - 1) / decode_arr
decode_throughput_mean, decode_throughput_std   = (
    decode_throughput_arr.mean(), decode_throughput_arr.std()
)

# overall throughput: all tokens over e2e latency
overall_throughput_arr                          = MAX_GENERATION_TOKENS / latency_arr
overall_throughput_mean, overall_throughput_std = (
    overall_throughput_arr.mean(), overall_throughput_arr.std()
)

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-25 00:22:19 [loggers.py:259] Engine 000: Avg prompt throughput: 52.0 tokens/s, Avg generation throughput: 112.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-25 00:22:29 [loggers.py:259] Engine 000: Avg prompt throughput: 45.0 tokens/s, Avg generation throughput: 128.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-25 00:22:39 [loggers.py:259] Engine 000: Avg prompt throughput: 40.0 tokens/s, Avg generation throughput: 127.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%


In [13]:
print(latency_times)

[1.1533946150000247, 1.1546663679999938, 1.15811794800004, 1.1581564630000685, 1.155379982999989, 1.1560436510000045, 1.1557793970000603, 1.1614258760000666, 1.1632449009999846, 1.1646089470000334, 1.167494425999962, 1.1672609199999897, 1.173931562000007, 1.1697113160000754, 1.1745459700000538, 1.1736473380000234, 1.1765040320000253, 1.1717623210000738, 1.1799025500000653, 1.1783765549999998, 1.1803409789999932, 1.1849738209999714, 1.18091124700004, 1.1860596709999527, 1.1843252630000052, 1.1860889650000672, 1.1896476570000232, 1.188914727999986, 1.1919414320000215, 1.1922308750000639]


In [14]:
print(ttft_times)

[0.026130423999916275, 0.02697804600006748, 0.02833562599994366, 0.026834669999971084, 0.022795677999965847, 0.02244531199994526, 0.02267763500003639, 0.026767943999971067, 0.026915727000073275, 0.026327163000019027, 0.027620542000022397, 0.02750148999996327, 0.02982273100008115, 0.027470228999959545, 0.028618038999979944, 0.027937307000001965, 0.028941030000055434, 0.02273227399996358, 0.026794535000021824, 0.02508178699997643, 0.023044484000024568, 0.02875008599994544, 0.023088568000048326, 0.02598124899998311, 0.025414997000098083, 0.022835484000097495, 0.025769054999955188, 0.02243317299996761, 0.02377041600004759, 0.021701258999996753]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"                        : MODEL_NAME,
    "model_type"                       : MODEL_TYPE,
    "compression_method"               : COMPRESSION_METHOD,
    #"sparsity"                         : SPARSITY,
    "input_token_count"                 : input_tokens,
    "generated_token_count"             : total_generated_tokens,
    "ttft_sec"                          : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "inference_sec"                     : f"{inference_mean:.2f} ± {inference_std:.2f}",
    "latency_sec"                       : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "decode_throughput_tokens_per_sec"  : f"{decode_throughput_mean:.2f} ± {decode_throughput_std:.2f}",
    "overall_throughput_tokens_per_sec" : f"{overall_throughput_mean:.2f} ± {overall_throughput_std:.2f}",
}

# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [16]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main c5cf343] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-GPTQ
 1 file changed, 12 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-GPTQ/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-GPTQ'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   5fbc077..c5cf343  main -> main
